# W3 보조 · 작은 예시 모음

본 노트북 없이도 개념을 빠르게 체험하는 **독립 mini 예제**들입니다. (helpers 폴더가 필요한 예제는 표시)

## 예제 1 · 평균의 함정 (학습 불필요, numpy만)

정답이 A일 수도 B일 수도 있을 때, L1 최적해 = 픽셀별 평균 = 회색.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
n=80; yy,xx=np.mgrid[0:n,0:n]
disk=lambda cx,cy,r: (((xx-cx)**2+(yy-cy)**2)<r*r).astype(float)
A=disk(28,40,14); B=disk(52,40,14)     # 두 후보
avg=0.5*(A+B)                          # L1 최적 = 평균
fig,ax=plt.subplots(1,3,figsize=(9,3.2))
for a,(im,t) in zip(ax,[(A,'후보 A'),(B,'후보 B'),(avg,'L1 평균 = 회색')]):
    a.imshow(im,cmap='gray_r',vmin=0,vmax=1); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()
print('평균(오른쪽)은 A도 B도 아닌 회색 두 개 → 진짜엔 없는 값.')

## 예제 2 · hinge 손실 직접 계산 (torch)

확실히 맞히면 벌점 0.

In [ ]:
import torch
for real,fake in [(1.5,-1.5),(0.8,0.3),(-0.2,0.9)]:
    d = torch.relu(torch.tensor(1.-real)) + torch.relu(torch.tensor(1.+fake))
    print(f'D(real)={real:+.1f} D(fake)={fake:+.1f}  ->  D 벌점 {float(d):.2f}')
print('진짜>=+1, 가짜<=-1 이면 벌점 0. 애매하거나 틀리면 벌점 발생.')

## 예제 3 · 판별자 점수 map 크기 (helpers 필요)

PatchGAN은 이미지 한 장에 점수 하나가 아니라 patch별 점수 map을 냅니다.

In [ ]:
import sys
from pathlib import Path
for _c in [Path('.'), Path('..')/'helpers']:
    if (_c/'model_utils.py').exists(): sys.path.insert(0,str(_c.resolve())); break
import torch
from model_utils import PatchDiscriminatorMini, count_parameters
D=PatchDiscriminatorMini(cond_ch=2, base=16)   # 생성자보다 작게 잡아 D 독주를 막음
for size in [64,128,256]:
    cond=torch.randn(1,2,size,size); y=torch.rand(1,1,size,size)
    print(f'입력 {size}x{size}  ->  점수 map {tuple(D(cond,y).shape)}')
print('파라미터:', f'{count_parameters(D):,}')

## 예제 4 · λ 하나 바꿔보기 (helpers + data 필요, 짧은 학습)

같은 조건(Bentheimer, k=2)에서 λ=0(회색) vs λ=0.2(선명)의 연속 출력을 비교.

In [ ]:
import numpy as np
from dr_utils import load_volume
from model_utils import train_gan, predict_continuous
DATA=next((p for p in [Path('data'),Path('..')/'data'] if (p/'Bentheimer_256.bin').exists()),Path('data'))
vol=load_volume(DATA/'Bentheimer_256.bin'); K=2; z=128; c0,c1=64,192
before,after,target=vol[z-K,c0:c1,c0:c1],vol[z+K,c0:c1,c0:c1],vol[z,c0:c1,c0:c1]
grey=lambda c:((c>0.2)&(c<0.8)).mean()
outs=[]
for lam in [0.0, 0.2]:
    G,_,_=train_gan(vol,k=K,lambda_gan=lam,epochs=12,warmup=3,d_base=16,device='cpu',verbose=False)
    outs.append((lam, predict_continuous(G,before,after)))
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,3,figsize=(9,3.2))
ax[0].imshow(target,cmap='gray_r',vmin=0,vmax=1); ax[0].set_title('원본'); ax[0].axis('off')
for a,(lam,c) in zip(ax[1:],outs):
    a.imshow(c,cmap='gray_r',vmin=0,vmax=1); a.set_title(f'lam={lam} · 회색 {grey(c)*100:.0f}%'); a.axis('off')
plt.tight_layout(); plt.show()